In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
import sqlite3

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
# %%
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://transparency.google/accountability/"
print("✅ Scraper do Google Transparency Report pronto!")

✅ Scraper do Google Transparency Report pronto!


In [3]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()


✅ Banco e tabela 'articles' prontos!


In [4]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [5]:
# %%
def load_articles_from_db():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT *
        FROM articles
        WHERE source = 'Google Transparency'
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles_from_db()
display(df_db.head())
print(f"📦 Total no banco: {len(df_db)} registros")


,id,title,date,author,url,source
0,3781,Learn about Transparency Reports,2026-05-02,Google,https://transparencyreport.google.com,Google Transparency
1,3782,TRANSPARENCY REPORTGoogle Safe BrowsingView ho...,2026-05-02,Google,https://transparencyreport.google.com/safe-bro...,Google Transparency
2,3783,TRANSPARENCY REPORTGovernment requests to remo...,2026-05-02,Google,https://transparencyreport.google.com/governme...,Google Transparency
3,3784,TRANSPARENCY REPORTYouTube Community Guideline...,2026-05-02,Google,https://transparencyreport.google.com/youtube-...,Google Transparency
4,3785,Explore all reports,2026-05-02,Google,https://transparencyreport.google.com/,Google Transparency


📦 Total no banco: 5 registros


In [6]:
# %%
def extrair_dados_relatorio(url):
    """Acessa a página do relatório e tenta extrair informações (embora seja SPA)."""
    # Como o site é um SPA, o requests puro não pega o conteúdo dinâmico.
    # Retornamos uma descrição padrão ou NA.
    return ["Google Transparency Report Details"]

def coletar_relatorios():
    print(f"📄 Coletando relatórios de: {BASE_URL}")
    try:
        r = requests.get(BASE_URL, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            print(f"⚠️ Erro HTTP {r.status_code}")
            return []
    except Exception as e:
        print(f"⚠️ Erro de conexão: {e}")
        return []

    soup = BeautifulSoup(r.text, "html.parser")
    relatorios = []
    
    # Encontrando os links de relatórios na página de accountability
    # Eles geralmente estão em blocos com link para transparencyreport.google.com
    links = soup.select("a[href*='transparencyreport.google.com']")
    
    for link_tag in links:
        url = link_tag.get("href")
        # Pegar o texto do pai ou do próprio link
        titulo = link_tag.get_text(strip=True)
        if not titulo or len(titulo) < 5:
            # Tentar pegar de um h3 ou h4 próximo
            parent = link_tag.find_parent("div")
            if parent:
                h = parent.find(["h3", "h4", "h2"])
                if h:
                    titulo = h.get_text(strip=True)
        
        if not titulo:
            continue
            
        # Data: Para o Google, os relatórios são periódicos. 
        # Tentar extrair do texto se houver (ex: "Sept 2024")
        data_match = re.search(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{4}', titulo + " " + link_tag.parent.get_text())
        data = data_match.group() if data_match else datetime.now().strftime("%Y-%m-%d")
        
        if url not in [r['url'] for r in relatorios]:
            relatorios.append({
                "titulo": titulo,
                "data": data,
                "url": url,
                "fonte": "Google Transparency"
            })
            
            # Inserir no banco
            insert_article(titulo, data, "Google", url, "Google Transparency")
            
    return relatorios

In [7]:
# %%
relatorios_coletados = coletar_relatorios()
print(f"✅ Total coletado: {len(relatorios_coletados)} relatórios")

df_google = pd.DataFrame(relatorios_coletados)
display(df_google.head())


📄 Coletando relatórios de: https://transparency.google/accountability/
✅ Total coletado: 5 relatórios


,titulo,data,url,fonte
0,Learn about Transparency Reports,2026-05-02,https://transparencyreport.google.com,Google Transparency
1,TRANSPARENCY REPORTGoogle Safe BrowsingView ho...,2026-05-02,https://transparencyreport.google.com/safe-bro...,Google Transparency
2,TRANSPARENCY REPORTGovernment requests to remo...,2026-05-02,https://transparencyreport.google.com/governme...,Google Transparency
3,TRANSPARENCY REPORTYouTube Community Guideline...,2026-05-02,https://transparencyreport.google.com/youtube-...,Google Transparency
4,Explore all reports,2026-05-02,https://transparencyreport.google.com/,Google Transparency


In [8]:
# %%
keywords = ['privacy', 'data', 'requests', 'removal', 'content', 'safety', 'security']
pattern = r'|'.join(keywords)

if not df_google.empty:
    df_filt = df_google[
        df_google['titulo'].str.contains(pattern, case=False, na=False, regex=True)
    ].copy()

    print(f"{len(df_filt)} relatórios filtrados (de {len(df_google)})")
    display(df_filt.head())

3 relatórios filtrados (de 5)


,titulo,data,url,fonte
1,TRANSPARENCY REPORTGoogle Safe BrowsingView ho...,2026-05-02,https://transparencyreport.google.com/safe-bro...,Google Transparency
2,TRANSPARENCY REPORTGovernment requests to remo...,2026-05-02,https://transparencyreport.google.com/governme...,Google Transparency
3,TRANSPARENCY REPORTYouTube Community Guideline...,2026-05-02,https://transparencyreport.google.com/youtube-...,Google Transparency
